# Proyecto de Ciencia de Datos: Rotten Tomatoes

In [ ]:
import kagglehub

base_path = kagglehub.dataset_download(
  "stefanoleone992/rotten-tomatoes-movies-and-critic-reviews-dataset",
  output_dir="./content/datasets",
  force_download=True
)

In [ ]:
critics_dataset_path = base_path + "/rotten_tomatoes_critic_reviews.csv"
movies_dataset_path = base_path + "/rotten_tomatoes_movies.csv"

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

In [ ]:
criticas = pd.read_csv(critics_dataset_path)
pelis = pd.read_csv(movies_dataset_path)

In [ ]:
print(pelis.shape)
print(criticas.shape)

In [ ]:
display(pelis.head())
display(criticas.head())

In [ ]:
pelis.info()

In [ ]:
criticas.info()

In [ ]:
def resumen_nulos(df):
    return pd.DataFrame({
        "nulos": df.isna().sum(),
        "nulos_%": (df.isna().mean() * 100).round(2),
        "unicos": df.nunique(dropna=True)
    }).sort_values("nulos_%", ascending=False)

resumen_nulos(pelis)

In [ ]:
resumen_nulos(criticas)

In [ ]:
print(pelis.duplicated().sum())
print(criticas.duplicated().sum())

In [ ]:
pelis_limpias = pelis.drop_duplicates().copy()
criticas_limpias = criticas.drop_duplicates().copy()

In [ ]:
for columna in pelis_limpias.select_dtypes(include="object").columns:
    pelis_limpias[columna] = pelis_limpias[columna].replace(r"^\s*$", np.nan, regex=True)

for columna in criticas_limpias.select_dtypes(include="object").columns:
    criticas_limpias[columna] = criticas_limpias[columna].replace(r"^\s*$", np.nan, regex=True)

In [ ]:
pelis_limpias["original_release_date"] = pd.to_datetime(pelis_limpias["original_release_date"], errors="coerce")
pelis_limpias["streaming_release_date"] = pd.to_datetime(pelis_limpias["streaming_release_date"], errors="coerce")
criticas_limpias["review_date"] = pd.to_datetime(criticas_limpias["review_date"], errors="coerce")

In [ ]:
pelis_limpias["release_year"] = pelis_limpias["original_release_date"].dt.year
pelis_limpias["main_genre"] = pelis_limpias["genres"].str.split(",").str[0].str.strip()
pelis_limpias["gap_audience_minus_critics"] = pelis_limpias["audience_rating"] - pelis_limpias["tomatometer_rating"]
criticas_limpias["review_year"] = criticas_limpias["review_date"].dt.year
criticas_limpias["is_fresh"] = criticas_limpias["review_type"].eq("Fresh").astype(int)

In [ ]:
print(pelis_limpias.shape)
print(criticas_limpias.shape)

In [ ]:
resumen_nulos(pelis_limpias)

In [ ]:
resumen_nulos(criticas_limpias)

In [ ]:
display(pelis_limpias.head())
display(criticas_limpias.head())

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

project_root = Path.cwd().resolve()
if not (project_root / "services").exists():
    project_root = project_root.parent

sys.path.append(str(project_root / "services" / "streaming_api"))

from data_service import candidate_data_dirs, canonical_platform, normalize_platform_columns, normalize_text, parse_runtime_minutes

plt.rcParams["figure.dpi"] = 140

In [ ]:
datasets_dir_candidates = [path for path in candidate_data_dirs() if path.exists()]
if not datasets_dir_candidates:
    raise FileNotFoundError("No se encontro ninguna carpeta de datasets. Ejecuta antes la descarga o revisa la ruta de trabajo.")

datasets_dir = next(
    (
        path
        for path in datasets_dir_candidates
        if list(path.glob("*_titles.csv")) and (path / "rotten_tomatoes_movies.csv").exists()
    ),
    datasets_dir_candidates[-1],
)
figures_dir = project_root / "cuadernos" / "content" / "figuras_modelo"
figures_dir.mkdir(parents=True, exist_ok=True)

platform_files = sorted(datasets_dir.glob("*_titles.csv"))
if not platform_files:
    raise FileNotFoundError(
        f"No se encontraron archivos *_titles.csv en {datasets_dir}. Verifica que esten Netflix, Hulu, Disney+ y Prime Video."
    )

platform_frames = []
platform_debug = []
for path in platform_files:
    platform = canonical_platform(path)
    if platform is None:
        platform_debug.append((path.name, "omitido", "plataforma no reconocida"))
        continue
    normalized = normalize_platform_columns(pd.read_csv(path), platform)
    if normalized.empty:
        platform_debug.append((path.name, platform, "sin columnas compatibles"))
        continue
    normalized["title_key"] = normalized["title"].map(normalize_text)
    normalized["release_year"] = pd.to_numeric(normalized["release_year"], errors="coerce")
    normalized["runtime_minutes"] = [
        parse_runtime_minutes(duration, content_type)
        for duration, content_type in zip(normalized["duration"], normalized["type"])
    ]
    platform_debug.append((path.name, platform, int(len(normalized))))
    platform_frames.append(normalized)

if not platform_frames:
    raise ValueError(
        "No se pudieron normalizar catalogos de plataformas. Diagnostico: "
        + "; ".join(f"{archivo} -> {detalle}" for archivo, _, detalle in platform_debug)
    )

platform_catalog = pd.concat(platform_frames, ignore_index=True)
platform_catalog = platform_catalog.drop_duplicates(subset=["title", "platform", "release_year"], keep="first")

rt_usecols = [
    "movie_title",
    "content_rating",
    "genres",
    "original_release_date",
    "runtime",
    "tomatometer_rating",
    "audience_rating",
    "tomatometer_count",
    "audience_count",
]
rt_catalog = pd.read_csv(datasets_dir / "rotten_tomatoes_movies.csv", usecols=rt_usecols)
rt_catalog["title_key"] = rt_catalog["movie_title"].map(normalize_text)
rt_catalog["release_year"] = pd.to_datetime(rt_catalog["original_release_date"], errors="coerce").dt.year
rt_catalog["runtime_minutes"] = pd.to_numeric(rt_catalog["runtime"], errors="coerce")
rt_catalog = rt_catalog.drop_duplicates(subset=["title_key", "release_year"], keep="first")

modelo_df = rt_catalog.merge(
    platform_catalog[["title_key", "release_year", "platform", "type", "country"]],
    on=["title_key", "release_year"],
    how="inner",
)
modelo_df = modelo_df.dropna(subset=["platform", "title_key"]).copy()
modelo_df["type"] = modelo_df["type"].fillna("Movie")
modelo_df["country"] = modelo_df["country"].fillna("Unknown")
modelo_df["age_rating"] = modelo_df["content_rating"]

features = [
    "type",
    "genres",
    "age_rating",
    "release_year",
    "runtime_minutes",
    "country",
    "tomatometer_rating",
    "audience_rating",
    "tomatometer_count",
    "audience_count",
]
numeric = [
    "release_year",
    "runtime_minutes",
    "tomatometer_rating",
    "audience_rating",
    "tomatometer_count",
    "audience_count",
]
categorical = ["type", "genres", "age_rating", "country"]

X = modelo_df[features]
y = modelo_df["platform"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical,
        ),
    ]
)
pipeline = Pipeline(
    [
        ("preprocess", preprocess),
        (
            "model",
            RandomForestClassifier(
                n_estimators=80,
                random_state=42,
                class_weight="balanced_subsample",
                n_jobs=-1,
            ),
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
classes = list(pipeline.named_steps["model"].classes_)
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
metricas_df = pd.DataFrame(report).T.loc[classes, ["precision", "recall", "f1-score"]]

resumen_catalogos = pd.DataFrame(platform_debug, columns=["archivo", "plataforma", "filas_normalizadas"])
display(resumen_catalogos)
print(modelo_df.shape)
print(y.value_counts())
display(metricas_df.round(3))

In [ ]:
def guardar_figura(fig, nombre):
    ruta = figures_dir / nombre
    fig.savefig(ruta, bbox_inches="tight", facecolor="white")
    print(ruta)
    plt.show()
    plt.close(fig)

accuracy = float((y_test == y_pred).mean())
metricas_resumen = pd.DataFrame(
    {
        "metrica": ["Accuracy", "Macro avg F1", "Weighted avg F1"],
        "valor": [
            accuracy,
            report["macro avg"]["f1-score"],
            report["weighted avg"]["f1-score"],
        ],
    }
)
display(metricas_resumen.round(3))

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(metricas_resumen["metrica"], metricas_resumen["valor"], color=["#8C4A3F", "#D98E5B", "#E6B566"])
ax.set_title("Resumen de desempeno del Random Forest")
ax.set_ylabel("Puntaje")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.2)
guardar_figura(fig, "modelo_metricas_resumen.png")

for metrica, nombre in [
    ("precision", "modelo_precision.png"),
    ("recall", "modelo_recall.png"),
    ("f1-score", "modelo_f1.png"),
]:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    metricas_df[metrica].sort_values(ascending=False).plot(kind="bar", ax=ax, color="#B35C4B")
    ax.set_title(f"{metrica.upper()} por plataforma")
    ax.set_xlabel("")
    ax.set_ylabel(metrica.upper())
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.2)
    guardar_figura(fig, nombre)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=classes,
    cmap="Oranges",
    xticks_rotation=20,
    ax=ax,
    colorbar=False,
)
ax.set_title("Matriz de confusion del Random Forest")
guardar_figura(fig, "modelo_matriz_confusion.png")

feature_names = pipeline.named_steps["preprocess"].get_feature_names_out()
importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": pipeline.named_steps["model"].feature_importances_,
    }
)

def origen_variable(nombre):
    if nombre.startswith("num__"):
        return nombre.replace("num__", "", 1)
    for columna in categorical:
        prefijo = f"cat__{columna}_"
        if nombre.startswith(prefijo):
            return columna
    return nombre

importance_df["variable"] = importance_df["feature"].map(origen_variable)
importancia_variables = (
    importance_df.groupby("variable", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)
top_componentes = importance_df.sort_values("importance", ascending=False).head(12).copy()
top_componentes["feature"] = top_componentes["feature"].str.replace("num__", "", regex=False)
top_componentes["feature"] = top_componentes["feature"].str.replace("cat__", "", regex=False)
display(importancia_variables.round(4))
display(top_componentes[["feature", "importance"]].round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(importancia_variables["variable"], importancia_variables["importance"], color="#D9C393")
ax.invert_yaxis()
ax.set_title("Importancia de variables del Random Forest")
ax.set_xlabel("Importancia acumulada")
guardar_figura(fig, "modelo_importancia_variables.png")

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(top_componentes["feature"], top_componentes["importance"], color="#B35C4B")
ax.invert_yaxis()
ax.set_title("Top 12 features mas influyentes del Random Forest")
ax.set_xlabel("Importancia")
guardar_figura(fig, "modelo_top_features.png")